## Database Connection (Config)

In [8]:
!pip install -q pymongo dnspython plotly pandas python-dotenv

import os
import json
from pymongo import MongoClient
from bson import json_util
from dotenv import load_dotenv

load_dotenv()

URI = os.getenv('MONGO_URI')
DB_NAME = os.getenv('DB_NAME', 'tracera')
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'raw_linkedin')

def connect_to_mongodb_and_get_data(uri, db_name, collection_name, query={}):
    if not uri:
        raise ValueError("Kredensial MONGO_URI tidak ditemukan! Pastikan file .env sudah ada.")
    try:
        client = MongoClient(uri)
        db = client[db_name]
        collection = db[collection_name]
        documents = list(collection.find(query))
        json_data = json.loads(json_util.dumps(documents))
        print(f"Berhasil terhubung! {len(json_data)} dokumen dimuat dari koleksi '{collection_name}'.")
        return json_data
    except Exception as e:
        print(f"Error koneksi: {e}")
        return []

raw_documents = connect_to_mongodb_and_get_data(URI, DB_NAME, COLLECTION_NAME)


Berhasil terhubung! 397 dokumen dimuat dari koleksi 'raw_linkedin'.


## Pre-processing Data

In [9]:
import pandas as pd

# 1. Taksonomi Soft Skills Terkurasi (25 Kata Kunci Standar)
SOFT_SKILLS = {
    'communication', 'communications', 'public speaking', 'presentation skills', 'komunikasi',
    'teamwork', 'collaboration', 'interpersonal skills', 'kerjasama tim', 'active listening',
    'leadership', 'team leadership', 'management', 'strategic planning', 'kepemimpinan',
    'problem solving', 'analytical skills', 'critical thinking', 'decision making', 'creativity',
    'time management', 'adaptability', 'negotiation', 'english', 'teaching'
}

# 2. Parsing Dokumen Mentah & Pembersihan Data Anomali
def clean_doc(d):
    raw_list = d.get('raw_data', [{}])
    p = raw_list[0] if isinstance(raw_list, list) and len(raw_list) > 0 else {}

    # a. Validasi & ekstrak riwayat pengalaman kerja (rentang tahun 1980 - 2027)
    experiences = []
    years = []
    for e in p.get('experience', []):
        yr = e.get('startDate', {}).get('year') if isinstance(e.get('startDate'), dict) else None
        if yr and str(yr).isdigit():
            yr_int = int(yr)
            if 1980 <= yr_int <= 2027:
                years.append(yr_int)
                experiences.append({'year': yr_int, 'title': (e.get('position') or '').strip()})

    # b. Standarisasi nama industri (fallback ke 'Other' jika kosong/null)
    ind = p.get('industry')
    ind_name = ind.get('name') if isinstance(ind, dict) else (ind or 'Other')
    ind_clean = ind_name.strip() if isinstance(ind_name, str) and ind_name.strip() else 'Other'

    # c. Deduplikasi Keahlian per Profil (Case-insensitive & buang spasi liar)
    skill_map = {}
    all_raw_s = [s['name'] for s in p.get('skills', []) if isinstance(s, dict) and s.get('name')]
    all_raw_s += (p.get('topSkills') or [])
    for s in all_raw_s:
        if s and isinstance(s, str):
            s_clean = s.strip()
            if len(s_clean) > 1:
                key = s_clean.lower()
                if key not in skill_map:
                    skill_map[key] = s_clean
                elif s_clean[0].isupper() and not skill_map[key][0].isupper():
                    skill_map[key] = s_clean

    unique_skills = list(skill_map.values())

    # d. Bersihkan nama program studi (buang noise '-', '.', dan deduplikasi dalam 1 profil)
    progs = []
    for e in p.get('education', []):
        fos = e.get('fieldOfStudy')
        if fos and isinstance(fos, str):
            fos_clean = fos.strip()
            if len(fos_clean) > 2 and fos_clean not in ['-', '.', '...', 'null', 'None']:
                if fos_clean not in progs:
                    progs.append(fos_clean)

    return {
        'id': p.get('publicIdentifier'),
        'industry': ind_clean,
        'skills': unique_skills,
        'programs': progs,
        'experiences': experiences,
        'latest_work_year': max(years, default=None)
    }

# 3. Buat DataFrame Utama & Hapus Duplikasi Profil Alumni
raw_parsed = [clean_doc(d) for d in raw_documents]
df_raw = pd.DataFrame(raw_parsed)
df_profiles = df_raw.drop_duplicates(subset=['id']).dropna(subset=['id']).reset_index(drop=True)

# 4. Explode Skills ke Baris Individual & Tagging Soft Skills
df_skills = df_profiles.explode('skills').dropna(subset=['skills']).copy()
df_skills['skills_clean'] = df_skills['skills'].str.strip()
df_skills['is_soft'] = df_skills['skills_clean'].str.lower().isin(SOFT_SKILLS)

# 5. Ringkasan Hasil Preprocessing
n_soft = int(df_skills['is_soft'].sum())
n_hard = int((~df_skills['is_soft']).sum())

print("Selesai Preprocessing!")
print(f"- Data mentah masuk : {len(raw_documents)} dokumen")
print(f"- Profil bersih unik: {len(df_profiles)} alumni (profil scraping ganda dibersihkan)")
print(f"- Total baris skill : {len(df_skills)} baris")
print(f"  • Hard skills     : {n_hard} baris ({round(n_hard/len(df_skills)*100, 1)}%)")
print(f"  • Soft skills     : {n_soft} baris ({round(n_soft/len(df_skills)*100, 1)}%)")

Selesai Preprocessing!
- Data mentah masuk : 397 dokumen
- Profil bersih unik: 394 alumni (profil scraping ganda dibersihkan)
- Total baris skill : 6113 baris
  • Hard skills     : 5431 baris (88.8%)
  • Soft skills     : 682 baris (11.2%)


## Dashboard Component 1: Hard Skills vs Soft Skills (Bar Chart)

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

selected_industry = ""  # Contoh lain: "Mining", "Oil and Gas", atau "" (Semua)
top_n = 10

# Filter Data
df_s_filt = df_skills[df_skills['industry'].str.lower() == selected_industry.lower()] if selected_industry else df_skills

# Ambil Top N Hard & Soft Skills
top_hard = df_s_filt[~df_s_filt['is_soft']]['skills'].value_counts().head(top_n).sort_values(ascending=True)
top_soft = df_s_filt[df_s_filt['is_soft']]['skills'].value_counts().head(top_n).sort_values(ascending=True)

# Subplots 1 Baris, 2 Kolom
fig_a = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"Top {top_n} Hard Skills", f"Top {top_n} Soft Skills"),
    shared_yaxes=False,
    horizontal_spacing=0.20
)

# Kiri: Hard Skills (Biru)
fig_a.add_trace(go.Bar(
    x=top_hard.values, y=top_hard.index,
    orientation='h', marker_color='#1d4ed8',
    name='Hard Skills', text=top_hard.values, textposition='auto'
), row=1, col=1)

# Kanan: Soft Skills (Hijau)
fig_a.add_trace(go.Bar(
    x=top_soft.values, y=top_soft.index,
    orientation='h', marker_color='#10b981',
    name='Soft Skills', text=top_soft.values, textposition='auto'
), row=1, col=2)

fig_a.update_layout(
    title_text=f"<b>a. Pemetaan Hard Skills & Soft Skills Alumni</b> — Industri: {selected_industry or 'Semua'}",
    height=420,
    showlegend=False,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig_a.show()

## Dashboard Component 2: Skills Distribution by Category

## Dashboard Component 3: Network of Top Skills

## Dashboard Component 4: Trending Skills

In [11]:
import re
from datetime import datetime, timezone
from collections import Counter, defaultdict
import pandas as pd
import plotly.graph_objects as go

# Parameter Konfigurasi Dashboard Trending Skills
top_n = 10              # Jumlah keahlian tren teratas yang ditampilkan
min_frequency = 5       # Ambang batas minimum frekuensi agar tren representatif
selected_industry = ""  # Filter industri, contoh: "Mining", "Oil and Gas", atau "" (Semua)

# 1. Ekstraksi Dokumen, Timestamp, dan Pembersihan Peran Pekerjaan (Linked Roles)
def clean_job_title(title):
    if not title or not isinstance(title, str):
        return ""
    t = title.split(' at ')[0].split(' - ')[0].split(' | ')[0].split(' @ ')[0].strip()
    low = t.lower()
    noise_keywords = ['experience', 'endorsed', 'across', 'grade:', 'passed', 'college', 'academy', 'universitas', 'gpa', 'alumni']
    if any(k in low for k in noise_keywords):
        return ""
    if len(t) < 3 or len(t) > 42:
        return ""
    return t

parsed_docs = []
for d in raw_documents:
    raw_list = d.get('raw_data', [{}])
    p = raw_list[0] if isinstance(raw_list, list) and len(raw_list) > 0 else {}
    pid = p.get('publicIdentifier')
    if not pid:
        continue

    # Filter Industri
    ind = p.get('industry')
    ind_name = ind.get('name') if isinstance(ind, dict) else (ind or 'Other')
    ind_clean = ind_name.strip() if isinstance(ind_name, str) and ind_name.strip() else 'Other'

    if selected_industry and ind_clean.lower() != selected_industry.lower():
        continue

    # Timestamp fetched_at dari metadata dokumen
    fa = d.get('metadata', {}).get('fetched_at')
    dt = datetime.fromisoformat(fa) if fa else datetime.now(timezone.utc)

    # Kumpulkan riwayat jabatan / peran kerja alumni
    exp_roles = []
    for e in p.get('experience', []):
        r = clean_job_title(e.get('position', ''))
        if r and r not in exp_roles:
            exp_roles.append(r)
    hl_role = clean_job_title(p.get('headline', ''))
    if hl_role and hl_role not in exp_roles:
        exp_roles.append(hl_role)

    # Ekstraksi keahlian unik per profil beserta posisi terkait
    skills_in_doc = []
    seen_skills = set()
    all_raw_skills = p.get('skills', []) if isinstance(p.get('skills'), list) else []
    for s in all_raw_skills:
        if isinstance(s, dict) and s.get('name'):
            s_name = s['name'].strip()
            if len(s_name) > 1 and s_name.lower() not in seen_skills:
                seen_skills.add(s_name.lower())
                # Peran kerja yang melekat langsung pada entri keahlian
                direct_roles = []
                for pos in s.get('positions', []):
                    cp = clean_job_title(pos)
                    if cp and cp not in direct_roles:
                        direct_roles.append(cp)
                skills_in_doc.append({'name': s_name, 'roles': direct_roles})

    if skills_in_doc:
        parsed_docs.append({
            'id': pid,
            'dt': dt,
            'skills': skills_in_doc,
            'profile_roles': exp_roles
        })

# 2. Pembagian Window Temporal (30 Hari Terakhir vs Baseline Sebelumnya)
parsed_docs.sort(key=lambda x: x['dt'])
mid_idx = len(parsed_docs) // 2
earlier_batch = parsed_docs[:mid_idx]
recent_batch = parsed_docs[mid_idx:]

n_earlier = max(len(earlier_batch), 1)
n_recent = max(len(recent_batch), 1)

skill_earlier = Counter()
skill_recent = Counter()
skill_linked_roles = defaultdict(Counter)

for item in earlier_batch:
    for s in item['skills']:
        s_name = s['name']
        skill_earlier[s_name] += 1
        for r in s['roles']:
            skill_linked_roles[s_name][r] += 1
        if not s['roles'] and item['profile_roles']:
            skill_linked_roles[s_name][item['profile_roles'][0]] += 1

for item in recent_batch:
    for s in item['skills']:
        s_name = s['name']
        skill_recent[s_name] += 1
        for r in s['roles']:
            skill_linked_roles[s_name][r] += 1
        if not s['roles'] and item['profile_roles']:
            skill_linked_roles[s_name][item['profile_roles'][0]] += 1

# 3. Hitung Tingkat Perubahan Minat / Popularitas (Momentum Share: Skala 0 - 100%)
# Mengukur persentase porsi adopsi keahlian yang muncul dalam 30 hari terakhir
all_unique_skills = set(skill_earlier.keys()) | set(skill_recent.keys())
trending_list = []

for s_name in all_unique_skills:
    c_rec = skill_recent[s_name]
    c_ear = skill_earlier[s_name]
    total_freq = c_rec + c_ear

    # Hanya ambil keahlian yang memenuhi ambang batas frekuensi dan mengalami peningkatan minat
    if total_freq >= min_frequency and c_rec > c_ear:
        # Momentum Share (0 - 100%): porsi kemunculan terkini terhadap total frekuensi
        momentum_pct = (c_rec / total_freq) * 100

        # Ambil maksimal 2 peran kerja teratas yang paling relevan
        top_roles = [r for r, _ in skill_linked_roles[s_name].most_common(2)]
        roles_text = ", ".join(top_roles) if top_roles else "Professional Specialist"

        trending_list.append({
            'Skill Name': s_name,
            'Frequency': total_freq,
            'Change in popularity (last 30 days)': momentum_pct,
            'Change_Formatted': f"+{momentum_pct:.1f}%",
            'Example linked roles': roles_text
        })

df_trending = pd.DataFrame(trending_list)
if not df_trending.empty:
    df_trending = df_trending.sort_values(
        by=['Change in popularity (last 30 days)', 'Frequency'],
        ascending=[False, False]
    ).head(top_n).reset_index(drop=True)
else:
    df_trending = pd.DataFrame(columns=['Skill Name', 'Frequency', 'Change in popularity (last 30 days)', 'Change_Formatted', 'Example linked roles'])

# 4. Bangun Komponen Tabel Ringkasan Interaktif (Plotly go.Table)
industry_label = selected_industry if selected_industry else 'Semua'

cell_colors = [
    ['#f8fafc' if i % 2 == 0 else '#ffffff' for i in range(len(df_trending))]
] * 4 if len(df_trending) > 0 else '#ffffff'

fig_d = go.Figure(data=[go.Table(
    columnwidth=[180, 90, 160, 310],
    header=dict(
        values=[
            '<b>Skill Name</b>',
            '<b>Frequency</b>',
            '<b>Change in popularity (last 30 days)</b>',
            '<b>Example linked roles</b>'
        ],
        fill_color='#0f172a',
        font=dict(color='white', size=12),
        align=['left', 'center', 'center', 'left'],
        height=36
    ),
    cells=dict(
        values=[
            df_trending['Skill Name'],
            df_trending['Frequency'],
            [f"<b>{val}</b>" for val in df_trending['Change_Formatted']],
            df_trending['Example linked roles']
        ],
        fill_color=cell_colors,
        font=dict(
            color=['#0f172a', '#1e293b', '#16a34a', '#334155'],
            size=11
        ),
        align=['left', 'center', 'center', 'left'],
        height=34
    )
)])

fig_d.update_layout(
    title_text=f"<b>d. Trending Skills: Peningkatan Popularitas Keahlian Teratas (30 Hari Terakhir)</b> — Industri: {industry_label}",
    margin=dict(l=20, r=20, t=60, b=20),
    height=420
)

fig_d.show()
